Build both Qdrant collections (DINO and QWEN) with payload_fields_to_index set up properly, so metadata filtering (e.g. "AP_WD between 3.0 and 3.4") is actually fast at scale instead of falling back to an unindexed scan.

Without payload_fields_to_index, Qdrant *can* still filter on these fields, but it has to scan payloads rather than use an index.

field_schema values come straight from Qdrant's own payload index types:
https://qdrant.tech/documentation/concepts/indexing/#payload-index
- "float"   -> numeric range filtering (>=, <=) -- use for all 12 settings
- "integer" -> numeric range filtering for whole numbers -- use for SEMImageID
- "keyword" -> exact-match / categorical filtering -- use for ID-like strings

In [ ]:
import os
import glob
import re
import io
import json
from PIL import Image
import numpy as np
import pandas as pd
import torch

from haystack import Document
from haystack.document_stores.types import DuplicatePolicy

from haystack_integrations.document_stores.qdrant import QdrantDocumentStore

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import base64

load_dotenv()

VLLM_API_KEY = os.getenv("VLLM_API_KEY")
VLLM_BASE_URL = "https://orfeo-llm.areasciencepark.it/vllm/v1"

client = OpenAI(base_url=VLLM_BASE_URL,api_key=VLLM_API_KEY)

GEMMA_MODEL = "google/gemma-4-26B-A4B-it"
DESCRIPTIONS_FILE = "subset_test_1000/image_descriptions.json"

At the moment, we are doing it locally (inside ./.qdrant).  
We create two collections, one for DINO and one for QWEN

In [ ]:
QDRANT_PATH = ".qdrant"
DINO_COLLECTION = "SEMImages_dino"
QWEN_COLLECTION = "SEMImages_qwen"
DINO_EMBEDDING_DIM = 768
QWEN_EMBEDDING_DIM = 4096

CROPS_FOLDER = "subset_test_1000/crops"
JSON_FOLDER = "subset_test_1000/json"
DINO_EMBEDDINGS_FOLDER = "subset_test_1000/embeddings"
QWEN_EMBEDDINGS_FOLDER = "subset_test_1000/embeddings_QWEN"

THUMBNAIL_SIZE = (224, 224)  # downsized preview for chat rendering, not a crop --
                              # full field of view preserved, only resolution drops.
                              # tune down to (128,128) if payload size becomes an issue.

These are the important keys to store from the JSON

In [ ]:
SETTINGS_KEYS = [
    "AP_WD",
    "AP_FREE_WD",
    "AP_BEAM_TIME",
    "AP_STAGE_LOW_Z",
    "AP_STAGE_CENTRE_X",
    "AP_C3",
    "AP_EHT_CAL_30KV",
    "AP_EHT_CAL_10KV",
    "AP_EHT_CAL_100",
    "AP_EHT_CAL_24KV",
    "AP_MINMAG",
]

# Matches: "<label> = <number> <unit>"  e.g. "WD = 3.198 mm"
_NUM_PATTERN = re.compile(r"=\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)\s*([^\s=]*)\s*$")

In [ ]:
def parse_value(raw_string):
    """
    Turn 'WD = 3.198 mm' into (3.198, 'mm').  
    Returns (None, None) if the string can't be parsed (e.g. field is None or empty in the source JSON).
    """
    if not raw_string:
        return None, None
    match = _NUM_PATTERN.search(raw_string)
    if not match:
        return None, None
    return float(match.group(1)), match.group(2)

In [ ]:
def load_all_metadata(folder_path):
    """
    Take the JSON folder and parse each file to extract the ID and the `SETTINGS_KEY`.  
    Returns a Pandas dataframe where each row is a file and columns are its values.
    """
    all_rows = []

    for filename in sorted(os.listdir(folder_path)):
        if not filename.endswith(".json"):
            continue

        file_path = os.path.join(folder_path, filename)
        with open(file_path, "r") as f:
            data = json.load(f)

        row = {
            "SEMImageID": data["SEMImageID"],
            "ImageOriginalName": data["ImageOriginalName"],
            "ImageHashName": data["ImageHashName"],
            "ImageAVGHashName": data["ImageAVGHashName"],
        }

        for key in SETTINGS_KEYS:
            value, unit = parse_value(data.get(key))
            row[key] = value # numeric column, e.g. AP_WD = 3.198
            row[f"{key}_unit"] = unit # e.g. AP_WD_unit = "mm"

        all_rows.append(row)

    df = pd.DataFrame(all_rows)
    print(f"Loaded {len(df)} files.")
    return df

In [ ]:
df = load_all_metadata("subset_test_1000/json")

In [ ]:
df.sort_values(by='SEMImageID', ascending=True).reset_index(drop=True).head()

In [ ]:
first_row = df.sort_values(by='SEMImageID', ascending=True).reset_index(drop=True).iloc[0]
first_row

In [ ]:
def load_embeddings(folder_path):
    """
    Take the embeddings (`.pt`) folder and load each file with `torch`.  
    Returns a dictionary of imageID: embedding.
    """
    embeddings = {}
    for filename in sorted(os.listdir(folder_path)):
        if not filename.endswith(".pt"):
            continue
        image_id = int(os.path.splitext(filename)[0])
        tensor = torch.load(os.path.join(folder_path, filename))
        embeddings[image_id] = tensor.flatten().tolist()
    return embeddings

In [ ]:
embeds = load_embeddings("subset_test_1000/embeddings_QWEN")

In [ ]:
dict(sorted(embeds.items(), key=lambda item: item[0]))

In [ ]:
def render_content(row):
    """
    Create content based on the metadata (from JSON).  
    Returns a string containing basic information about the image (not used during retrieval, only used by LLM to generate answer).
    """
    parts = [f"SEM image {row['SEMImageID']}."]
    if row["AP_WD"] is not None:
        parts.append(f"Working distance: {row['AP_WD']} {row['AP_WD_unit']}.")
    if row["AP_MINMAG"] is not None:
        parts.append(f"Minimum magnification: {row['AP_MINMAG']} {row['AP_MINMAG_unit']}.")
    if row["AP_C3"] is not None:
        parts.append(f"C3 lens current: {row['AP_C3']} {row['AP_C3_unit']}.")
    if row["AP_BEAM_TIME"] is not None:
        parts.append(f"Beam time: {row['AP_BEAM_TIME']} {row['AP_BEAM_TIME_unit']}.")
    return " ".join(parts)

In [ ]:
render_content(first_row)

In [ ]:
def encode_image(path, size=None):
    arr = np.load(path)
    image = Image.fromarray(arr, mode="L")
    if size is not None:
        image = image.resize(size, Image.LANCZOS)
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{b64}"

In [ ]:
def generate_image_description(image_path):
    """
    Ask Gemma 4 to describe a SEM image.
    """
    image_url = encode_image(image_path)

    response = client.chat.completions.create(
        model=GEMMA_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an expert in scanning electron microscopy. "
                    "Describe the image focusing on morphology, structure, "
                    "particle shape, size, texture, and visible patterns. "
                    "Do not invent information that is not visible."
                )
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "Describe this SEM image. "
                            "Provide a concise scientific description."
                        )
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": image_url
                        }
                    }
                ]
            }
        ],
        temperature=0.2,
    )

    return response.choices[0].message.content

In [ ]:
def load_descriptions():
    if os.path.exists(DESCRIPTIONS_FILE):
        with open(DESCRIPTIONS_FILE, "r") as f:
            return json.load(f)
    return {}

def save_descriptions(descriptions):
    with open(DESCRIPTIONS_FILE, "w") as f:
        json.dump(
            descriptions,
            f,
            indent=2
        )

In [ ]:
def get_crop_path(image_id):
    pattern = os.path.join(
        CROPS_FOLDER,
        f"{image_id}_*.npy"
    )

    matches = glob.glob(pattern)

    if len(matches) == 0:
        raise FileNotFoundError(
            f"No crop found for image ID {image_id}"
        )

    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple crops found for image ID {image_id}: {matches}"
        )

    return matches[0]

In [ ]:
def get_description(image_id):

    descriptions = load_descriptions()

    key = str(image_id)

    if key in descriptions:
        return descriptions[key]

    crop_path = get_crop_path(image_id)

    print(f"Generating description for image {image_id}")

    description = generate_image_description(crop_path)

    descriptions[key] = description

    save_descriptions(descriptions)

    return description

In [ ]:
def build_meta(row, include_thumbnail=False):
    """
    Build the metadata for filtering to append to the Document to upload to Qdrant.  
    Returns a dictionary of key: value.
    """
    meta = {
        "SEMImageID": int(row["SEMImageID"]),
        "ImageOriginalName": row["ImageOriginalName"],
        "ImageHashName": row["ImageHashName"],
    }
    for key in SETTINGS_KEYS:
        value = row[key]
        if value is not None:
            meta[key] = float(value)

    if include_thumbnail:
        crop_path = get_crop_path(int(row["SEMImageID"]))
        meta["thumb_b64"] = encode_image(crop_path, size=THUMBNAIL_SIZE)
    return meta

In [ ]:
def build_documents(embeddings, json_folder, include_thumbnail=False):
    """
    Build the Haystack Document to upload to Qdrant.  
    Returns a list of Documents.
    """
    df = load_all_metadata(json_folder)
    documents = []
    skipped = []

    for _, row in df.iterrows():
        image_id = row["SEMImageID"]
        if image_id not in embeddings:
            skipped.append(image_id)
            continue

        description = get_description(image_id)

        content = render_content(row)

        content += (
            "\n\nVisual description:\n"
            + description
        )

        documents.append(
            Document(
                id=str(image_id),
                content=content,
                embedding=embeddings[image_id],
                meta=build_meta(row, include_thumbnail=include_thumbnail),
            )
        )

    if skipped:
        print(f"Warning: {len(skipped)} image(s) had metadata but no embedding, skipped: {skipped[:10]}...")
    return documents

In [ ]:
get_description(111)

In [ ]:
docs = build_documents(embeds, JSON_FOLDER, True)

In [ ]:
docs[0].meta

All 12 settings are numeric (parsed as float in load_metadata.py), so they all get the same "float" schema -- this is what makes range filters like `{"AP_WD": {">=": 3.0, "<=": 3.4}}` fast instead of an unindexed scan.

In [ ]:
PAYLOAD_FIELDS_TO_INDEX = [
    {"field_name": key, "field_schema": "float"} for key in SETTINGS_KEYS
] + [
    {"field_name": "SEMImageID", "field_schema": "integer"},
]

In [ ]:
def build_collection(collection_name, embeddings_folder, embedding_dim, json_folder):
    if embedding_dim is None:
        raise ValueError(
            f"embedding_dim not set for collection '{collection_name}' -- "
            f"fill in the actual output dimension before running."
        )

    document_store = QdrantDocumentStore(
        index=collection_name,
        embedding_dim=embedding_dim,
        recreate_index=True,
        payload_fields_to_index=PAYLOAD_FIELDS_TO_INDEX,
        path=QDRANT_PATH, # local
        # url=QDRANT_URL, # server
    )

    embeddings = load_embeddings(embeddings_folder)
    documents = build_documents(embeddings, json_folder)
    document_store.write_documents(documents, policy=DuplicatePolicy.OVERWRITE)
    print(f"Wrote {len(documents)} documents to '{collection_name}' (dim={embedding_dim}), "
          f"indexed on {len(PAYLOAD_FIELDS_TO_INDEX)} payload fields.")
    return document_store

In [ ]:
document_store = build_collection(QWEN_COLLECTION, QWEN_EMBEDDINGS_FOLDER, QWEN_EMBEDDING_DIM, JSON_FOLDER)

In [ ]:
doc = document_store.get_documents_by_id(["111"])
doc[0].content

In [ ]:
document_store.count_documents_by_filter({"field": "meta.SEMImageID", "operator": "<=", "value": 200})

In [ ]:
def render_image_gallery(documents) -> str:
    """
    Render the retrieved candidates as inline images so the researcher can
    actually see what's being discussed, not just read a paraphrase of it.
    """
    if not documents:
        return ""

    gallery = "\n\n**Candidate images:**\n"
    for idx, doc in enumerate(documents, start=1):
        label = doc.meta.get("ImageOriginalName", doc.id)
        thumb = doc.meta.get("thumb_b64")
        gallery += f"\n{idx}. {label}\n"
        if thumb:
            gallery += f"\n![{label}]({thumb})\n"
    return gallery

In [ ]:
render_image_gallery([docs[0]])